# Name That LLM: tuning the fine-tuned encoder with Optuna

**Goal:** the [encoder-select notebook](https://www.kaggle.com/code/ryqn00/name-that-llm-encoder-select)
compared two encoders at their usual recommended settings. ModernBERT-base won with seed-7
CV 0.344, but DeBERTa-v3-base (0.206) clearly underfit: its loss barely moved at its gentle
default learning rate. So this notebook first gives DeBERTa a **fair rematch** at
ModernBERT's settings, then tunes the true winner. Optuna searches the three knobs that
matter most when fine-tuning:
- the **learning rate**: too high wrecks the pretrained weights, too low learns nothing;
- the number of **epochs**;
- the **warmup**: how long the learning rate ramps up from 0.

Part of the learning-in-public MLOps project
[github.com/Ryqn0/name-that-llm](https://github.com/Ryqn0/name-that-llm), plan P5.

**Each trial is a GPU fine-tuning run, so the search is built to be cheap:**
- **Proxy folds:** a trial trains on 2 of the 5 seed-7 folds (0 and 1), not all 5. The
  winner is re-measured on full, fresh CV in the final notebook.
- **Warm start:** trial 0 is always the default config, so tuning can't end below it.
- **Pruning:** after its first fold, a trial below the median of earlier trials stops there.
  A bad configuration costs one fold instead of two.
- **Fixed epochs:** the number of epochs is a tuned hyperparameter, never early-stopped on a
  validation fold (that would leak the fold into its own score).

In [ ]:
# Pinned to an exact commit, so this run is reproducible even after main moves on.
%pip install -q "name-that-llm[tune,track] @ git+https://github.com/Ryqn0/name-that-llm@02e0917b93bba5209135b6f702b2aa947cbe1db9"

In [ ]:
import os
import time

import torch
import transformers
import yaml

from name_that_llm.config import EncoderConfig, load_config
from name_that_llm.data import load_train
from name_that_llm.encoder import run_cv_encoder
from name_that_llm.tune import best_encoder_config, run_encoder_study

print(
    "torch",
    torch.__version__,
    "| transformers",
    transformers.__version__,
    "| GPUs",
    [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
)
assert torch.cuda.is_available(), "this notebook needs a GPU (accelerator: T4)"

# Tracking is optional: without the Kaggle Secret, everything still runs.
try:
    from kaggle_secrets import UserSecretsClient

    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    TRACK = True
except Exception as exc:
    TRACK = False
    print(f"W&B tracking is OFF ({type(exc).__name__}). Results are still computed.")

train = load_train()

## 1. A fair rematch: DeBERTa at ModernBERT's settings

The same seed-7 folds and the same settings that worked for ModernBERT: learning rate 5e-5,
batch 16 (DeBERTa's memory peak at batch 8 was 3.5 GB, so 16 fits). Whichever model scores
higher is the one we tune.

In [ ]:
MODERNBERT_CV = 0.3441  # ModernBERT-base's seed-7 CV, from the encoder-select notebook
FAIR = {"lr": 5e-5, "batch_size": 16, "seed": 7}  # ModernBERT's settings, the same folds
deberta = load_config("deberta_v3_base", model_cls=EncoderConfig).model_copy(update=FAIR)
deberta_cv = run_cv_encoder(deberta, train).score
print(
    f"DeBERTa-v3-base at lr 5e-5, batch 16: seed-7 CV {deberta_cv:.4f} "
    f"(ModernBERT-base: {MODERNBERT_CV:.4f})"
)

if deberta_cv > MODERNBERT_CV:
    WINNER, BASE = "deberta_v3_base", deberta
else:
    WINNER, BASE = "modernbert_base", load_config("modernbert_base", model_cls=EncoderConfig)
print(f"tuning {WINNER}, starting from: {BASE}")

## 2. The search: 10 trials, at most 2 folds each

Optuna prints one line per finished trial. A **pruned** trial stopped after its first fold
because it was already below the median.

In [ ]:
N_TRIALS = 10
start = time.time()
study = run_encoder_study(BASE, train, n_trials=N_TRIALS, seed=7)
print(f"{len(study.trials)} trials in {(time.time() - start) / 60:.0f} min")

trials = study.trials_dataframe()[
    ["number", "state", "value", "params_lr", "params_epochs", "params_warmup_ratio"]
]
# print(), not just the display: the downloadable run log only keeps printed text.
print(trials.round(6).to_string(index=False))

default, best = study.trials[0], study.best_trial
print(f"default (trial 0): {default.value:.4f} with {default.params}")
print(f"best (trial {best.number}): {best.value:.4f} with {best.params}")

## 3. The tuned config

In [ ]:
tuned = best_encoder_config(study)
print(yaml.safe_dump(tuned.model_dump(), sort_keys=False))
with open("/kaggle/working/encoder.yaml", "w") as f:
    yaml.safe_dump(tuned.model_dump(), f, sort_keys=False)
trials.to_csv("/kaggle/working/encoder_trials.csv", index=False)
print("outputs:", sorted(os.listdir("/kaggle/working")))

if TRACK:
    import wandb

    run = wandb.init(
        project="name-that-llm",
        job_type="encoder-tune",
        name="encoder-tune",
        config=tuned.model_dump(),
    )
    run.summary.update(
        {
            "default_value": default.value,
            "best_value": best.value,
            "n_pruned": int((trials["state"] == "PRUNED").sum()),
        }
    )
    run.finish()

## 4. Findings

*Filled in after the run.*